# Expected Wait

### Definition
The proportion of the time during the Day Part in which an arriving passenger would experience an Actual Headway that is greater than the Scheduled Headway.

### Reporting Grain
Service Pattern, stop, Day Part, and Reporting Period

### Notes
Also including expected wait time matching Chicago Transit Authority performance metrics.

In [1]:
import os
from sqlalchemy import create_engine
from jinja2 import Environment, FileSystemLoader
import pandas as pd
import geopandas as gpd
import shapely.wkt

from src.data_loading import (
    get_matching_shapes_feeds,
    get_schedule_rt_headways,
    get_schedule_feed_entries,
)
from src.metric_calculation import calculate_expected_wait

### Parameters

In [2]:
TARGET_DATES = [
    "2026-06-15","2026-06-16","2026-06-17","2026-06-18","2026-06-19"
]
TARGET_ROUTE_SHORT_NAME = os.environ.get("TARGET_ROUTE_SHORT_NAME", "ECR")
TARGET_SCHEDULE_FEED = os.environ.get("TARGET_SCHEDULE_FEED", "Bay Area 511 SamTrans Schedule")
CONSTANT_MACROS = {
    "TARGET_DATES": TARGET_DATES,
    "TARGET_ROUTE_SHORT_NAME": TARGET_ROUTE_SHORT_NAME,
    "TARGET_SCHEDULE_FEED": TARGET_SCHEDULE_FEED
}
CONSTANT_MACROS

{'TARGET_DATES': ['2026-06-15',
  '2026-06-16',
  '2026-06-17',
  '2026-06-18',
  '2026-06-19'],
 'TARGET_ROUTE_SHORT_NAME': 'ECR',
 'TARGET_SCHEDULE_FEED': 'Bay Area 511 SamTrans Schedule'}

In [3]:
# min count of trips per service period to be considered reliable
MIN_TRIPS = 10

### Implementation

In [4]:
# gcp boilerplate
CALITP_BQ_MAX_BYTES = os.environ.get("CALITP_BQ_MAX_BYTES", 5_000_000_000)
CALITP_BQ_LOCATION = os.environ.get("CALITP_BQ_LOCATION", "us-west2")
engine = create_engine(
    f"bigquery://cal-itp-data-infra/?maximum_bytes_billed={CALITP_BQ_MAX_BYTES}",  # noqa: E231
    location=CALITP_BQ_LOCATION
)
env = Environment(loader=FileSystemLoader("templates"))


In [5]:
# Get the target route's service patterns, their geometry, and the feeds behind them
matching_shapes, feed_dates, feed_keys = get_matching_shapes_feeds(
    engine, env, CONSTANT_MACROS, min_trips=MIN_TRIPS
)
display(feed_dates)
matching_shapes

,service_date,feed_key
0,2026-06-15,4dfefaf518766d1cfc3d254a65e4295e
1,2026-06-18,d71492afe126cd61f629b17c3c056fcd
2,2026-06-17,d71492afe126cd61f629b17c3c056fcd
4,2026-06-16,d71492afe126cd61f629b17c3c056fcd
5,2026-06-19,d71492afe126cd61f629b17c3c056fcd


,feed_key,last_seen_date,gtfs_dataset_name,shape_id,route_id,route_short_name,ct,geometry
0,4dfefaf518766d1cfc3d254a65e4295e,2026-06-15,Bay Area 511 SamTrans Schedule,ECR2069,ECR,ECR,350,"LINESTRING (-122.16599 37.44395, -122.16578 37..."
1,4dfefaf518766d1cfc3d254a65e4295e,2026-06-15,Bay Area 511 SamTrans Schedule,ECR2068,ECR,ECR,350,"LINESTRING (-122.46879 37.70620, -122.46897 37..."
2,d71492afe126cd61f629b17c3c056fcd,2026-06-19,Bay Area 511 SamTrans Schedule,ECR2069,ECR,ECR,350,"LINESTRING (-122.16599 37.44395, -122.16578 37..."
3,d71492afe126cd61f629b17c3c056fcd,2026-06-19,Bay Area 511 SamTrans Schedule,ECR2068,ECR,ECR,350,"LINESTRING (-122.46879 37.70620, -122.46897 37..."


In [6]:
# Get the schedule feed entry for every feed that was active during the window
feed_dates = get_schedule_feed_entries(engine, env, feed_dates)
feed_dates

,service_date,feed_key,base64_url,_valid_from
0,2026-06-15,4dfefaf518766d1cfc3d254a65e4295e,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-05-16 03:00:00.573378+00:00
1,2026-06-18,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
2,2026-06-17,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
3,2026-06-16,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
4,2026-06-19,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00


In [7]:
# Get the realtime headway between every pair of consecutive trips, at every
# stop on the pattern rather than only the timepoints
schedule_rt_headways = get_schedule_rt_headways(
    engine,
    env,
    CONSTANT_MACROS,
    feed_dates=feed_dates,
    shape_ids=matching_shapes.shape_id.to_list(),
)
schedule_rt_headways


,service_date,first_trip_key,second_trip_key,stop_sequence,stop_id,shape_id,first_actual_departure,second_actual_departure,rt_headway_seconds,first_scheduled_departure_sec,second_scheduled_departure_sec,scheduled_headway_seconds,pt_geom
0,2026-06-15,c8c8031789732fd986518ec203acde2a,f9b18b7a5eef9cd86713db5af47a749e,1,332014,ECR2068,2026-06-15 18:15:00,2026-06-15 18:27:00,720,65400,66300,900,POINT(-122.468854 37.70622)
1,2026-06-15,f9b18b7a5eef9cd86713db5af47a749e,3712792ffc168bd236a9488e43012b6c,1,332014,ECR2068,2026-06-15 18:27:00,2026-06-15 19:46:00,4740,66300,71100,4800,POINT(-122.468854 37.70622)
2,2026-06-15,3712792ffc168bd236a9488e43012b6c,4433aa6310aa729627cad33b98110fbd,1,332014,ECR2068,2026-06-15 19:46:00,2026-06-15 22:18:00,9120,71100,80100,9000,POINT(-122.468854 37.70622)
3,2026-06-15,7f2779002a08bac8c4013694a5d7c267,c8c8031789732fd986518ec203acde2a,2,332170,ECR2068,2026-06-15 17:43:55,2026-06-15 18:16:42,1967,63775,65562,1787,POINT(-122.462519 37.70498)
4,2026-06-15,c8c8031789732fd986518ec203acde2a,f9b18b7a5eef9cd86713db5af47a749e,2,332170,ECR2068,2026-06-15 18:16:42,2026-06-15 18:30:42,840,65562,66462,900,POINT(-122.462519 37.70498)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
34696,2026-06-18,3ca0efd2eb313a529d99c3ad3a619f65,ebe776fef9731d85a9c9d8fe2b271972,75,332019,ECR2069,2026-06-18 21:24:00,2026-06-18 21:31:00,420,76740,77640,900,POINT(-122.468737 37.706122)
34697,2026-06-18,ebe776fef9731d85a9c9d8fe2b271972,2e4cd72b3bf47e83ccb6605c8b96165d,75,332019,ECR2069,2026-06-18 21:31:00,2026-06-18 21:44:00,780,77640,78540,900,POINT(-122.468737 37.706122)
34698,2026-06-18,2e4cd72b3bf47e83ccb6605c8b96165d,9c88df7a0b055a24ec223400d24eeeaa,75,332019,ECR2069,2026-06-18 21:44:00,2026-06-18 22:21:00,2220,78540,80340,1800,POINT(-122.468737 37.706122)
34699,2026-06-18,9c88df7a0b055a24ec223400d24eeeaa,2af105a17ad2ed813e009a5d60cf9d54,75,332019,ECR2069,2026-06-18 22:21:00,2026-06-18 23:10:00,2940,80340,83220,2880,POINT(-122.468737 37.706122)


In [8]:
schedule_rt_headways.loc[(schedule_rt_headways.stop_sequence == 2) & (schedule_rt_headways.first_actual_departure.dt.hour == 14)].sort_values(by="first_actual_departure")

,service_date,first_trip_key,second_trip_key,stop_sequence,stop_id,shape_id,first_actual_departure,second_actual_departure,rt_headway_seconds,first_scheduled_departure_sec,second_scheduled_departure_sec,scheduled_headway_seconds,pt_geom
8560,2026-06-16,1e5e64de925a318351c2ca22f534c487,e34222ef5927fb0ea09a182e0181c0de,2,347016,ECR2069,2026-06-16 14:07:21,2026-06-16 14:23:21,960,50781,51681,900,POINT(-122.168262 37.444456)
3161,2026-06-16,592c12750d35312527fc3b24c6eb5694,1fe05bd25f052f83a02b99127e86ee49,2,332170,ECR2068,2026-06-16 14:12:42,2026-06-16 14:27:42,900,51162,52062,900,POINT(-122.462519 37.70498)
8561,2026-06-16,e34222ef5927fb0ea09a182e0181c0de,dddc5e163fe6acb05376e9137349eb2c,2,347016,ECR2069,2026-06-16 14:23:21,2026-06-16 14:37:21,840,51681,52581,900,POINT(-122.168262 37.444456)
3162,2026-06-16,1fe05bd25f052f83a02b99127e86ee49,c76467417f84e7e72b842608c9257e06,2,332170,ECR2068,2026-06-16 14:27:42,2026-06-16 14:45:42,1080,52062,52962,900,POINT(-122.462519 37.70498)
8562,2026-06-16,dddc5e163fe6acb05376e9137349eb2c,bd059e6ebbd84c7b02033980a53914b3,2,347016,ECR2069,2026-06-16 14:37:21,2026-06-16 14:53:21,960,52581,53481,900,POINT(-122.168262 37.444456)
3163,2026-06-16,c76467417f84e7e72b842608c9257e06,b7830e7ab18ef2fbf2199cb19167fbb2,2,332170,ECR2068,2026-06-16 14:45:42,2026-06-16 14:58:42,780,52962,53862,900,POINT(-122.462519 37.70498)
8563,2026-06-16,bd059e6ebbd84c7b02033980a53914b3,a76ebc8439dee9d5ba13699b2915e327,2,347016,ECR2069,2026-06-16 14:53:21,2026-06-16 15:07:34,853,53481,54394,913,POINT(-122.168262 37.444456)
3164,2026-06-16,b7830e7ab18ef2fbf2199cb19167fbb2,9ab6143152446fc3d84b7010677cf763,2,332170,ECR2068,2026-06-16 14:58:42,2026-06-16 15:13:55,913,53862,54775,913,POINT(-122.462519 37.70498)
19081,2026-06-17,e3e056faa6260ff79221bfb352de1088,7a600b471ec440bf0faec14788563da2,2,347016,ECR2069,2026-06-17 14:07:21,2026-06-17 14:21:21,840,50781,51681,900,POINT(-122.168262 37.444456)
13587,2026-06-17,d098ec90e4b145c804aea86ed7ca08b5,f5fc3de894390b6ef733eeded2d980ab,2,332170,ECR2068,2026-06-17 14:13:42,2026-06-17 14:42:42,1740,51162,52962,1800,POINT(-122.462519 37.70498)


In [9]:
# The query carries the headway itself, so the distribution is ready to look at
schedule_rt_headways["rt_headway_seconds"].describe()

count    34701.000000
mean      1006.716089
std        597.948576
min          0.000000
25%        720.000000
50%        960.000000
75%       1212.000000
max      18060.000000
Name: rt_headway_seconds, dtype: float64

### Results

In [10]:
# Expected and excess wait per stop and day part, pooled over every selected date.
#
# Each headway is assigned to a day part by the clock hour it opens on, so a
# trip the schedule publishes as 24:15 counts in Owl the following morning.
# Day parts a stop saw no service in are absent rather than present and empty.
headway_summary = calculate_expected_wait(schedule_rt_headways)
headway_summary

,shape_id,stop_sequence,stop_id,time_period,n_headways,mean_rt_headway_seconds,median_rt_headway_seconds,max_rt_headway_seconds,rt_headway_variance,mean_scheduled_headway_seconds,median_scheduled_headway_seconds,scheduled_expected_wait_seconds,rt_expected_wait_seconds,excess_wait_seconds,percent_time_excess_wait
0,ECR2068,1,332014,Early AM,16,1515.000000,1290.0,3360.0,5.253600e+05,1522.500000,1260.0,630.0,930.886139,300.886139,0.014851
1,ECR2068,1,332014,AM Peak,18,2133.333333,1740.0,5520.0,2.377365e+06,2113.333333,1800.0,900.0,1623.861520,723.861520,0.028125
2,ECR2068,1,332014,Midday,11,4969.090909,3060.0,18060.0,2.539067e+07,4990.909091,2700.0,1350.0,5039.406047,3689.406047,0.012075
3,ECR2068,1,332014,PM Peak,16,4537.500000,3150.0,13260.0,1.744554e+07,4518.750000,3150.0,1575.0,4191.123554,2616.123554,0.035537
4,ECR2068,1,332014,Evening,6,3940.000000,3600.0,7320.0,4.408800e+06,3900.000000,3600.0,1800.0,2529.492386,729.492386,0.015228
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
775,ECR2069,75,332019,Early AM,9,1713.333333,1860.0,2280.0,3.106000e+05,1600.000000,1800.0,900.0,947.308690,47.308690,0.101167
776,ECR2069,75,332019,AM Peak,33,1000.000000,960.0,2940.0,3.237000e+05,1047.272727,900.0,450.0,661.850000,211.850000,0.132727
777,ECR2069,75,332019,Midday,56,957.857143,900.0,2580.0,2.104971e+05,928.928571,900.0,450.0,588.807766,138.807766,0.217002
778,ECR2069,75,332019,PM Peak,71,902.535211,960.0,2040.0,2.239649e+05,945.633803,900.0,450.0,575.343047,125.343047,0.171348


In [11]:
headway_summary.loc[headway_summary.time_period == "Midday"].describe()

,stop_sequence,n_headways,mean_rt_headway_seconds,median_rt_headway_seconds,max_rt_headway_seconds,rt_headway_variance,mean_scheduled_headway_seconds,median_scheduled_headway_seconds,scheduled_expected_wait_seconds,rt_expected_wait_seconds,excess_wait_seconds,percent_time_excess_wait
count,156.000000,156.000000,156.000000,156.000000,156.000000,1.560000e+02,156.000000,156.000000,156.000000,156.000000,156.000000,156.000000
mean,39.807692,57.294872,989.833971,953.567308,2353.461538,4.231393e+05,983.905875,940.384615,470.192308,613.503263,143.310955,0.173022
std,22.938942,5.709322,502.864659,398.565404,1476.167948,2.250864e+06,505.059563,387.178186,193.589093,452.806459,305.390302,0.056987
min,1.000000,10.000000,878.052632,780.000000,1560.000000,4.354983e+04,867.152542,900.000000,450.000000,472.302361,22.302361,0.001043
25%,20.000000,57.000000,915.700000,900.000000,1905.000000,1.204177e+05,915.517241,900.000000,450.000000,529.682224,79.682224,0.134153
50%,39.500000,58.000000,935.661978,900.000000,2220.000000,1.804730e+05,926.847458,900.000000,450.000000,559.361426,109.361426,0.176163
75%,59.250000,59.000000,946.374773,930.000000,2520.000000,2.409734e+05,933.973684,900.000000,450.000000,597.164205,147.164205,0.213970
max,82.000000,61.000000,5754.000000,5400.000000,18060.000000,2.539067e+07,5760.000000,5400.000000,2700.000000,5039.406047,3689.406047,0.382478


In [12]:
# Total headways observed per day
# note fewer headways on start and end of the week
schedule_rt_headways["first_actual_departure"].dt.date.value_counts()

first_actual_departure
2026-06-17    10662
2026-06-18    10508
2026-06-16    10430
2026-06-15     3101
Name: count, dtype: int64

In [13]:
# The longest gaps in service, which are what riders actually notice
(
    schedule_rt_headways
    .sort_values("rt_headway_seconds", ascending=False)
    .head(20)
    [["shape_id", "stop_sequence", "stop_id", "first_actual_departure", "second_actual_departure", "rt_headway_seconds"]]
)

,shape_id,stop_sequence,stop_id,first_actual_departure,second_actual_departure,rt_headway_seconds
3117,ECR2068,1,332014,2026-06-16 11:11:00,2026-06-16 16:12:00,18060
29681,ECR2069,1,347609,2026-06-18 05:07:00,2026-06-18 10:06:00,17940
13547,ECR2068,1,332014,2026-06-17 18:12:00,2026-06-17 21:53:00,13260
29684,ECR2069,1,347609,2026-06-18 15:36:00,2026-06-18 19:07:00,12660
24213,ECR2068,1,332014,2026-06-18 18:11:00,2026-06-18 21:18:00,11220
8534,ECR2069,1,347609,2026-06-16 19:25:00,2026-06-16 22:23:00,10680
19050,ECR2069,1,347609,2026-06-17 16:36:00,2026-06-17 19:25:00,10140
19045,ECR2069,1,347609,2026-06-17 09:51:00,2026-06-17 12:37:00,9960
13541,ECR2068,1,332014,2026-06-17 11:26:00,2026-06-17 14:11:00,9900
29683,ECR2069,1,347609,2026-06-18 12:51:00,2026-06-18 15:36:00,9900


In [14]:
# Map of one day part
#
# Both service patterns are drawn as plain lines for context. The metric lives on
# the stops, coloured by how much of the day part would have left a rider waiting
# longer than the schedule promised.
MAP_TIME_PERIOD = "Midday"

# The headways query carries each stop's point geometry, so the metric maps
# straight onto it - no projection onto the shape is needed the way it is for
# the segment metric in trip_duration_variability
stops = (
    schedule_rt_headways[["shape_id", "stop_sequence", "stop_id", "pt_geom"]]
    .drop_duplicates()
    .dropna(subset=["pt_geom"])
)
stops_gdf = gpd.GeoDataFrame(
    stops.drop(["pt_geom"], axis=1),
    geometry=gpd.GeoSeries(stops["pt_geom"].map(shapely.wkt.loads), crs=4326),
)

stops_for_period = stops_gdf.merge(
    headway_summary.loc[headway_summary["time_period"] == MAP_TIME_PERIOD],
    on=["shape_id", "stop_sequence", "stop_id"],
    how="inner",
)

# percent_time_excess_wait is a proportion; the map reads better in percent. The
# colour and the tooltip use the one column, so the legend matches what is hovered
stops_for_period["percent_time_excess_wait"] = (
    stops_for_period["percent_time_excess_wait"] * 100
).round(1)
for column in [
    "scheduled_expected_wait_seconds",
    "rt_expected_wait_seconds",
    "excess_wait_seconds",
]:
    stops_for_period[column] = stops_for_period[column].round(1)

# the shapes underneath, uncoloured. a shape carries the same geometry in every
# feed version it appears in, so draw each one once
shape_lines = matching_shapes.drop_duplicates(subset=["shape_id"])[
    ["shape_id", "geometry"]
]
period_map = shape_lines.explore(
    color="#888888",
    tooltip=["shape_id"],
    tiles="CartoDB positron",
    name="service patterns",
)
stops_for_period.explore(
    m=period_map,
    column="percent_time_excess_wait",
    cmap="plasma",
    marker_kwds={"radius": 6},
    style_kwds={"stroke": False, "fillOpacity": 0.9},
    tooltip=[
        "shape_id",
        "stop_sequence",
        "stop_id",
        "scheduled_expected_wait_seconds",
        "rt_expected_wait_seconds",
        "excess_wait_seconds",
        "percent_time_excess_wait",
    ],
    tooltip_kwds={
        "aliases": [
            "Shape",
            "Stop sequence",
            "Stop",
            "Scheduled expected wait (s)",
            "Actual expected wait (s)",
            "Excess wait (s)",
            "% of time waiting over schedule",
        ]
    },
    legend_kwds={
        "caption": f"% of {MAP_TIME_PERIOD.lower()} spent waiting longer than scheduled"
    },
    name=f"{MAP_TIME_PERIOD} excess wait",
)
period_map.save(f"expected_wait_{MAP_TIME_PERIOD.lower()}.html")
period_map